# Figure 8

In [ ]:
from geosage.plotting.paper import render_data_fit
from geosage.paths import workspace_root, data_path, result_path
ROOT = workspace_root()

from pathlib import Path
import csv
import json

import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 18,
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'axes.unicode_minus': False,
    'axes.linewidth': 0.8,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
})

ROOT = workspace_root()
FIGURE_DIR = ROOT / 'outputs' / 'figures'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CASE_SPECS = {
    'Hannah': {'inversion_dir': 'Hannah_Inversion_GPT'},
    'Iowa': {'inversion_dir': 'Iowa_Inversion_GPT'},
}
COMBINED_OUTPUT = FIGURE_DIR / 'Figure8_datafit_assessment.png'
PROPERTIES = {
    'gravity': {'label': 'Gravity', 'prediction_file': 'dpred_gravity.npy'},
    'magnetics': {'label': 'Magnetics', 'prediction_file': 'dpred_magnetics.npy'},
}


def load_property(case, property_name):
    """Load coordinates, observations, predictions, residuals, and units."""
    inversion_dir = result_path(CASE_SPECS[case]['inversion_dir'], ROOT)
    observed_data = np.loadtxt(inversion_dir / 'observed_data' / f'{property_name}.obs', ndmin=2)
    predicted = np.load(inversion_dir / 'inversion_result' / PROPERTIES[property_name]['prediction_file']).reshape(-1)
    if observed_data.shape[1] < 4 or predicted.size != observed_data.shape[0]:
        raise ValueError(f'{case} {property_name}: observation and prediction arrays are not aligned.')
    with (inversion_dir / 'inversion_params.json').open(encoding='utf-8') as handle:
        gravity_component = json.load(handle).get('gravity_component', '').lower()
    unit = 'nT' if property_name == 'magnetics' else ('E' if gravity_component == 'gzz' else 'mGal')
    observed = observed_data[:, 3]
    residual = predicted - observed
    if not (np.isfinite(observed).all() and np.isfinite(predicted).all()):
        raise ValueError(f'{case} {property_name}: non-finite values found.')
    return observed_data[:, :2], observed, predicted, residual, unit


def render_combined_comparison():
    columns = (('Hannah', 'gravity', 'Hannah ISO'), ('Hannah', 'magnetics', 'Hannah TMI'),
               ('Iowa', 'gravity', 'Iowa Gzz'), ('Iowa', 'magnetics', 'Iowa TMI'))
    return render_data_fit(columns, load_property, COMBINED_OUTPUT)

def fit_metrics(case, property_name):
    _, observed, predicted, residual, unit = load_property(case, property_name)
    rmse = np.sqrt(np.mean(residual ** 2))
    ss_total = np.sum((observed - np.mean(observed)) ** 2)
    return {
        'case': case,
        'property': PROPERTIES[property_name]['label'],
        'unit': unit,
        'n': observed.size,
        'bias_predicted_minus_observed': np.mean(residual),
        'mae': np.mean(np.abs(residual)),
        'rmse': rmse,
        'nrmse_percent_of_observed_range': 100 * rmse / np.ptp(observed),
        'r_squared': 1 - np.sum(residual ** 2) / ss_total,
        'pearson_r': np.corrcoef(observed, predicted)[0, 1],
    }

combined_figure = render_combined_comparison()
metrics = [fit_metrics(case, property_name) for case in CASE_SPECS for property_name in PROPERTIES]
metrics_csv = FIGURE_DIR / 'Figure8_datafit_assessment_metrics.csv'
with metrics_csv.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(metrics[0]))
    writer.writeheader()
    writer.writerows(metrics)

print(f'Wrote {combined_figure.relative_to(ROOT)}')
print(f'Wrote {metrics_csv.relative_to(ROOT)}')
for metric in metrics:
    print(metric)
